# Bell's Theorem in the Quantum Ising Model

$$\newcommand{\ket}[1]{\left|{#1}\right\rangle}$$
$$\newcommand{\bra}[1]{\left\langle{#1}\right|}$$

## Background

Previously, we worked with the classical Ising Hamiltonian

$$ \mathcal{H}_\mathrm{classical} = -J \sum_{\langle j,k \rangle} s_j s_k - h \sum_j s_j. $$

in terms of the classical spin variables $s_j$ which could be either "up" ($+1$) or "down" ($-1$); we will now consider a model of quantum spin-1/2 particles described by the Pauli spin matrices $\sigma^\alpha$ for $\alpha \in \{x,y,z\}$. The spins interact by trying to align the $z$-components of their spins, while simultaneously trying to align the $x$-component of their spins along an external magnetic field; hence, this model is often referred to as the quantum Ising model or the transverse-field Ising model (TFIM). This model is described by the quantum Hamiltonian:

$$ \mathcal{H} = -J \sum_{\langle j,k \rangle} \sigma_j^z \sigma_k^z - h \sum_j \sigma_j^x. $$

Recall that the quantum state of this system is described by a vector; since each spin can have two quantum states (up/down), the state of $N$ spins altogether will be a vector with $2^N$ components. This Hamiltonian is now a matrix which multiplies these vectors. In order to find the ground state of the system and its corresponding ground state energy, we'll need to find the eigenvalues and eigenvectors of the Hamiltonian matrix.

Looking at the first piece of the Hamiltonian: each Pauli matrix can have an eigenvalue of $\pm 1$, so this reproduces the behavior of the classical Ising model: when both spins have the same eigenvalue for their $z$-component of spin, the energy is lowered by $-J$, but if they have opposite eigenvalues, the energy is increased by $+J$. The second term introduces the quantum mechanics to this system.

### Frustration, Uncertainty, and Entanglement
The second term in the Hamiltonian lowers the energy by $-h$ when the $x$-component of the spin has eigenvalue $+1$ and raises the energy by $+h$ with eigenvalue $-1$; however, we've seen before that measuring a quantum system with a particular operator forces the state of the system to become one of the eigenvectors of that operator. The eigenvectors of $\sigma^z$ are $\pmatrix{1 & 0}^\top$ and $\pmatrix{0 & 1}^\top$, while the eigenvectors of $\sigma^x$ are $\frac1{\sqrt{2}}\pmatrix{1 & 1}^\top$ and $\frac1{\sqrt{2}}\pmatrix{1 & -1}^\top$, so the two terms in our Hamiltonian are trying to collapse the spin states into different vectors at the same time. This competing interest is commonly referred to as **frustration** and is the key underlying factor for a wide variety of interesting quantum physics in models of magnetism. 

Another view of this problem relates to the **Heisenberg Uncertainty Principle**, which you have maybe heard of in relation to measuring a particle's position and momentum. Mathematically, the statement is often given as 

$$ \Delta x \Delta p \geq \hbar $$

where $\Delta x$ and $\Delta p$ are the uncertainties in measuring position and momentum, respectively. This places a bound on how precisely I can simultaneously measure these two quantities; for example, if I know the position of a particle very exactly, then I can't know it's momentum at the same time. If I do a measurement to gain knowledge about momentum, then I have to lose information about the position as a tradeoff. There is a very similar relationship between different components of spin; because $\sigma^z$ and $\sigma^x$ have different sets of eigenvectors, I can't measure the eigenvalue of both at the same time; if I know one and try to measure the other, then I lose the information about the first one when I gain the information for the second.

As a result of this frustration between measuring $z$- and $x$-components of spin at the same time, the compromise nature comes up with is entanglement; in general, the quantum state of my Ising model will be a complicated superposition of spin states with correlations between measurements of spins at different sites that can't be explained classically.

In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
%matplotlib inline

import matplotlib as mpl
mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('xtick.minor', visible=True)
mpl.rc('ytick.minor', visible=True)

## Magnetization and Phase Diagram

Your first task: identify the quantum phase transition in the TFIM. The functions below create the Hamiltonian matrix $\mathcal{H}$ for the TFIM and the operator $M^z = \frac1{N} \sum_{j=1}^N \sigma_j^z$ for the average $z$-magnetization. Start with a system with $N=4$ spins.

1. Use `np.logspace` to create an array of values for the transverse field $h$ which spans values from $10^{-2}$ to $10^2$.
2. For each value of $h$ in your array, compute the Hamiltonian, and find its eigenvalues and eigenvectors. Identify the eigenvector which corresponds to the ground state and compute the expectation value of the operator $M^z$ in this state. Store $\langle M^z \rangle$ in an array. (Hint: do this in a `for` loop.)
3. Create a nice plot of $\langle M^z \rangle$ vs. $h$; you'll probably want to use `plt.semilogx` for this, since the array of $h$ values is log-spaced. Include a dashed horizontal line at the maximum magnetization $+1$.
4. Referring back to the classical simulation if needed, take a numerical derivative of your magnetization vs. $h$ to find the value of $h$ where the derivative has the greatest magnitude. We'll call this our estimate for the critical value of $h_c$. Make a plot of the derivative vs. $h$, and add dashed vertical lines to both plots denoting the position of $h_c$. 

In [ ]:
Id = np.array([[1,0],[0,1]]);
PauliX = np.array([[0,1],[1,0]]);
PauliY = np.array([[0, -1j],[1j,0]]);
PauliZ = np.array([[1,0],[0,-1]]);

def single2full(op, j, N):
    ## Converts the operator op in a single-spin Hilbert space
    ## to a single-site operator in an N-spin Hilbert space
    ## at site j

    oplist = [Id for j in range(N)];
    oplist[j-1] = op;

    Op = oplist[0];
    for k in range(1,N):
        Op = np.kron(Op, oplist[k]);

    return Op

def H_Zeeman(nSites, hvec):
    ## creates the Hamiltonian for an external magnetic field 
    ## in the direction hvec for a system with nSites

    ## create lists of operators at each site
    xList = [single2full(PauliX, j, nSites) for j in range(nSites)];
    yList = [single2full(PauliY, j, nSites) for j in range(nSites)];
    zList = [single2full(PauliZ, j, nSites) for j in range(nSites)];

    Ham = np.zeros((2**nSites, 2**nSites));
    ## add terms for field at each site
    for j in range(nSites):
        Ham = Ham -hvec[0] * xList[j] - hvec[1] * yList[j] - hvec[2] * zList[j];

    return Ham

def H_Ising(nSites, J=1., Periodic=True):
    ## creates Hamiltonian a 1D spin-1/2 chain with Ising bonds
    ## `Periodic` can be set to True or False to set boundary conditions;
    ## the default is to make the system periodic.

    ## create list of operators at each site
    zList = [single2full(PauliZ, j, nSites) for j in range(nSites)];

    Ham = np.zeros((2**nSites, 2**nSites));
    ## add bond terms connecting site j to site j+1
    for j in range(nSites-1):
        Ham = Ham - J*(zList[j] @ zList[j+1]);

    ## Periodic Boundary: add bond connecting N to 1 
    if Periodic:
        Ham = Ham - J*(zList[nSites-1] @ zList[0]);

    return Ham

def H_TFIM(nSites, hvec, J=1., Periodic=True):
    ## creates Hamiltonian for a TFIM with field given by the array hvec
    return H_Ising(nSites, J, Periodic) + H_Zeeman(nSites, hvec)

def Mz(nSites):
    ## creates the operator for the average z-magnetization in a system
    ## of nSites
    zList = [single2full(PauliZ, j, nSites) for j in range(nSites)];

    M = np.zeros((2**nSites, 2**nSites));
    for j in range(nSites):
        M = M + zList[j];

    return M/nSites

In [ ]:
## example of setup for Hamiltonian and M; finding ground state and <Mz>
N = 4;

hx = 0.3; hz = 1e-6;
hvec = np.array([hx, 0, hz]);

## No need to specify J or Periodic, we'll just use the default values to start
H = H_TFIM(N, hvec);
M = Mz(N);

## Solve for eigvals, eigvecs of H
eigvals, eigvecs = la.eig(H);
## Identify ground state energy and ground state
E0 = np.min(eigvals);
psi0 = eigvecs[:, eigvals==E0];
psi0 = psi0.T[0]; 
## transpose operation needed to fix shape of eigenvector,
## and to make the eigenvector a 1d vector, not a (2^N by 1) 2d array

## compute <ψ0 | Mz | ψ0>
magnetization = np.inner(psi0, M@psi0);
print("Average z-magnetization: ", magnetization)

### Quantum vs. Classical Phase Transitions

In the classical Ising model, we saw that there was a transition between a ferromagnet and a paramagnet at a particular temperature $T_C$. This transition was driven by the presence of thermal fluctuations in the magnet; when the temperature was high enough, the thermal fluctuations are strong enough to destroy the ordered ferromagnet state and form a disordered paramagnet. This is quite similar to the more familiar solid-liquid phase transition, where once again thermal fluctuations in the movement of atoms in a crystal eventual overcome the ordered, lattice structure and melt the crystal into a liquid. 

In a quantum phase transition, we are looking at the ground state of the system at absolute zero temperature, so there are no thermal fluctuations to worry about. Instead, the ordered ferromagnet is destroyed by quantum fluctuations, driven by an external parameter. In this case, the parameter is $h$, which relates the frustration of the field in the $x$-direction competing with the bonds between $z$-component of spins. You can understand this roughly through the action of $\sigma_x$ on a eigenstate of $\sigma_x$; if a spin is in the state $\ket{\uparrow}$, then multiplying this state by $\sigma_x$ flips the state to $\ket{\downarrow}$. As I increase the strength of the field along $x$, these spin flips eventually win out over the desire to align the spins along their $z$-components.

In an infinite 1D TFIM, the transition happens at exactly $h_c=J$. For $h<J$, the system is a ferromagnet, with spins mostly aligned along up or down in the $z$-direction. For $h>J$, the system is described more by a state of spins pointing along the $x$-direction, which looks like a paramagnet if we measure along $z$, since spins pointing along $+\hat{x}$ have a 50/50 chance of measuring up/down along $\hat{z}$; this is also sometimes referred to as the polarized phase, since the spins are being pushed to align along the external field.

### Symmetry and Degeneracy

Similar to the classical Ising case, we once again have a symmetry in the ground state of the ferromagnetic phase, where the system can randomly pick to align more along the up direction, or along down, and both have the same energy and are valid ground states. We refer to these states with equal energy as **degenerate**. In numerical situations, degeneracies can often lead to confusing outcomes; for example, in the limit $h\ll J$, a state which is an equal superposition of all spins up and all spins down, written as 

$$ \frac1{\sqrt{2}} (\ket{\uparrow \uparrow \cdots \uparrow} + \ket{\downarrow \downarrow \cdots \downarrow}) $$

is also a valid ground state (you can check the energy if you like). This superposition is sometimes called a **cat state**, as in Schrodinger's cat, since it is equal parts "alive" (all spins up) and "dead" (dead). We would never find this state in a real system, because thermal fluctuations at any finite temperature will push the system to break the symmetry and pick one state over the other (and all real experiments happen at finite temperature).

In order to avoid cat states and other superpositions of degenerate ground states, we can simply break the symmetry by adding in a small field in the $z$-direction to bias the spins towards up or down, similar to the classical case. For the rest of this notebook, make sure to keep a small $z$ field in all the calculations to ensure we don't run into any weirdness due to degeneracy.

## CHSH Violation

We're now going to investigate the behavior of the CHSH quantities we studied earlier in the context of the TFIM. Before, we started with a Bell pair state and then computed the CHSH expectation values $\langle S_1 \rangle$ and $\langle S_2 \rangle$ for different values of the angle $\theta$ between the axes of spin measurements. Now, we want to use a ground state of the TFIM as our starting state, instead of the Bell pair, and see how $\langle S_1 \rangle$ and $\langle S_2 \rangle$ change as we change both the angle $\theta$ and the magnetic field $h$. 

To start, fix $\theta = \pi/4$ for the angle between measurement axes and consider a TFIM with only $N=2$ sites so that one site is Alice's qubit, and the other is Bob's (which will be rotated before measurement). We can later try to generalize this setup to systems with more sites, time permitting. I've copied some of the CHSH code below to get you started. 

1. Create an array of $h$ values linearly spaced from 0 to 2. 
2. As before, compute the Hamiltonian and its eigenvalues/eigenvectors for each value of $h$. Identify the ground state, rotate Bob's qubit, and compute the necessary expectation values for $\langle S_1 \rangle$ and $\langle S_2 \rangle$, storing these in an array for plotting. 
3. Create a nice plot of $\langle S_1 \rangle$ and $\langle S_2 \rangle$ vs. $h$. Include appropriate dashed horizontal lines at $\{2, 2\sqrt{2} \}$ to distinguish where the CHSH inequality is violated and the maximal violation.
4. Use your setup from above to estimate the critical field value $h_c$ for a system with $N=2$ spins, and add a dashed vertical line at this value on your CHSH plot.

Once you've got this up and running, try changing the angle $\theta$ and seeing how that changes the behavior. Do different values of $\theta$ make bigger CHSH violation for different values of $h$? Does $\langle S_1 \rangle$ or $\langle S_2 \rangle$ capture the transition better for different $\theta$?

In [ ]:
def Ry(theta):
    '''
    Returns 2-qubit gate which rotates the second qubit by an 
    an angle theta (in radians) about the y-axis.
    '''
    Ry_single = np.array([[np.cos(theta/2), -np.sin(theta/2)],[np.sin(theta/2),np.cos(theta/2)]])
    return np.kron(Id, Ry_single);

In [ ]:
## Example of CHSH calculation for one h value
N = 2;

hx = 0.3; hz = 1e-6;
hvec = np.array([hx, 0, hz]);

## Create H
H = H_TFIM(N, hvec);

## Solve for eigvals, eigvecs of H
eigvals, eigvecs = la.eig(H);
## Identify ground state energy and ground state
E0 = np.min(eigvals);
psi0 = eigvecs[:, eigvals==E0];
psi0 = psi0.T[0]; 
## transpose operation needed to fix shape of eigenvector,
## and to make the eigenvector a 1d vector, not a (2^N by 1) 2d array

## Rotate Bob's measurement bases
theta = np.pi/4;
psi0_rot = Ry(theta)@psi0;

## operators for CHSH measurements
XX = np.kron(PauliX, PauliX);
XZ = np.kron(PauliX, PauliZ);
ZX = np.kron(PauliZ, PauliX);
ZZ = np.kron(PauliZ, PauliZ);

## Computing expectation values in the state psi0_rot
E_XX = np.inner(psi0_rot, XX@psi0_rot);
E_XZ = np.inner(psi0_rot, XZ@psi0_rot);
E_ZX = np.inner(psi0_rot, ZX@psi0_rot);
E_ZZ = np.inner(psi0_rot, ZZ@psi0_rot);

## Compute the two CHSH quantities
S1 = E_XX + E_XZ - E_ZX + E_ZZ;
S2 = E_XX - E_XZ + E_ZX + E_ZZ;

print("First CHSH quantity: ", np.abs(S1))
print("Second CHSH quantity: ", np.abs(S2))

## The Importance of Saving Outputs

Once you have all the codes above working as intended and can produce nice plots from the output, you'll want to start making plots to be used on your poster presentation. Once you've started producing presentation-quality data, it's important to save your data so that (1) you don't have to rerun the same code over and over to view a particular result and (2) you can easily share the raw data with collaborators or publish it alongside your results. When saving data, it's also *extremely* important to make sure you know what set of parameters created the code; a good file name for your data should likely include a short description of what the data IS, any relevant parameters used in making the data, and possibly a date and/or time. You can organize your data however you like, but I cannot stress enough how important it is to have SOME sort of system in place for you to organize your data, since you will likely eventually be in situations where you produce a LOT of data sets for the same kinds of calculations.

 - For saving and loading the raw data here, a simple way that gets the job done is to use the functions `np.savetxt` and `np.loadtxt`. You should read their documentation, but the short version is that `np.savetxt` takes in a filename (as a string), and the 2D array you want to save, and then creates a `.txt` file in the same folder as your Jupyter notebook with the array saved. You can also use additional options to change the format of the text file (i.e. tab delimited vs. comma delimited, adding header rows, etc.). When you want to load the data in, you just run `np.loadtxt` with the appropriate filename input and you'll recover the array exactly as you saved it. This also helpful for making the plots with data sets from multiple simulation runs on the same axes, since you can just load the data for each run in as a separate variable, then plot everything together.
 - For saving plots, screenshots are usually pretty effective, but if you want something more precise (or higher resolution), I would recommend `plt.savefig`, which just requires you to input options for the filename, name of the figure you want saved, file format (I'd usually just use `.png` for posters), and the dpi (I usually use at least a dpi of 200 for a poster, since the image will likely get printed larger than it is on your screen).

## Further Exploration

If you have spare time before needing to wrap things up and make your poster, here are some other things you could investigate in this model. Feel free to add code cells as needed, and remember to save outputs and plots for any results you'd like to hold onto or potentially present.

1. Collect and save data for the magnetization vs. $h$ for varying numbers of sites; you can probably go up to about $N=10$ or $12$ before the eigensolver is painfully slow. Plot all of these on the same plot (like the very last plot in the Julia Demo linked below), and make a new plot with your estimate of $h_c$ as a function of $N$; you should find that for larger $N$, $h_c$ moves closer to the exact value of $1$. This procedure of studying the critical behavior for different size systems is called **finite size scaling** and is something you may see a lot in research in this field.
2. In the Hamiltonian, change `Periodic` to `False` and compare all your results side-by-side with the `True` results; I suspect this won't change anything substantially, but it's good to check.
3. Pick a few values of $\theta$, collect the CHSH data, then plot $\langle S_1 \rangle$ and $\langle S_2 \rangle$ for different $\theta$ on the same axes to compare. (Maybe start with angles from $0$ to $2\pi$, spaced every $\pi/4$ or every $\pi/8$)
4. Increase the field in the $z$-direction `hz` and see how this changes your results; make a new plot showing how $h_c$ moves as a function of `hz`. I would suggest checking `hz` values spaced logarithmically from $10^{-3}$ up to about $10^1$.
5. Turn on the temperature; let me know if you get to this point and I'll walk you through modifying the code to get a finite temperature state instead of just the zero temperature ground state. From there, we could construct a colormap plot showing $\langle M^z \rangle$ as a function of both $h$ and $T$, and investigate how temperature effects the CHSH quantity. I could also talk you through computing all the other regular thermodynamic quantities like energy, heat capacity, entropy, etc. as functions of both $h$ and $T.$
6. For the CHSH inequality to work, the eigenvalues of the measurement operators must be $\pm 1$. For systems with more than 2 spins, I have a few ideas of how to generalize it. For one, we could simply assign Alice and Bob single qubits from the chain (i.e. Alice measures site 1, Bob measures site 6) and then investigate how the behavior changes as you change which qubits they are assigned. For a second approach, we could split the chain in half, assigning one half to Alice and the other to Bob, and create some new normalized operator that measures the average magnetization on both halves. (We could also use other operators with different eigenvalues, but then rederive the appropriate CHSH bounds.)

## Resources

 - [Quantum Ising Wiki](https://en.wikipedia.org/wiki/Transverse-field_Ising_model#Phases_of_the_1D_transverse_field_Ising_model)
 - [Julia Demo](https://juliaphysics.github.io/PhysicsTutorials.jl/tutorials/general/quantum_ising/quantum_ising.html)